In [1]:
# !pip install selenium
# !pip install hdfs pandas

In [2]:
# !dir

In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By

from bs4 import BeautifulSoup as bs
from hdfs import InsecureClient
import pandas as pd
from datetime import datetime
import time as tt

options = Options()
# options.binary_location = './driver_u/......' # 리눅스용
# options.binary_location = r'D:\data\jupyter_data\driver_w\chrome.exe' # 크롬위치, 윈도우용, r은 raw string 로우 문자열
options.add_argument("--headless")
# options.add_argument("--no-sandbox") # windows에서는 없어도 괜찮다.
# options.add_argument("--disable-dev-shm-usage") # 공유 메모리 비활성화
# options.add_argument("--disable-gpu") # GPU 비활성화
options.add_argument("--window-size=1920,1080") # 가상 화면 크기 설정

service = Service(r'D:\data\jupyter_data\driver_w\chromedriver.exe')

driver = webdriver.Chrome(service=service, options=options )
driver.get('https://tenpercentcoffee.com/')
print(driver.title)

hover_element = driver.find_element(By.CSS_SELECTOR ,"#header > div")

def gogo(sel):
    driver.find_element(By.CSS_SELECTOR , sel).click()

#마우스 호버액션
action = ActionChains(driver)
action.move_to_element(hover_element).perform()
print('메뉴 여는중...')
# 시간대기

tt.sleep(1)
gogo("#header > div > div > ul > li:nth-child(4) > ul > li > a")
print('매장 위치검색중...')

tt.sleep(2)
print('매장 위치로드완료')
gogo("#q")
tt.sleep(1)
driver.find_element(By.CSS_SELECTOR , "#q" ).send_keys("서울특별시")
driver.find_element(By.CSS_SELECTOR , "#q" ).send_keys(Keys.ENTER)
tt.sleep(1)

텐퍼센트커피
메뉴 여는중...
매장 위치검색중...
매장 위치로드완료


In [4]:
tenp = []
html = driver.page_source
soup = bs(html, 'html.parser')
contents = soup.select("#list_area > div.rlw div a ")
print('추출중..')

추출중..


In [5]:
data = [[
    c.select_one(".rlwbp.idfont1").text,
    c.select_one(".rlwsp").text,
    c['onclick'].replace('setMove(','').split(',')[0],
    c['onclick'].replace('setMove(','').split(',')[1]] for c in contents]
df = pd.DataFrame(data, columns=['name','location','lat','lon'])
df

,name,location,lat,lon
0,가락시장역점,서울특별시 서울 송파구 송파대로 28길 27 102동 109호,37.4948868947464,127.120093192726
1,가산대성디폴리스점,서울특별시 금천구 서부샛길 606 대성디폴리스,37.4799534974765,126.877008239669
2,가산디지털단지역점,서울특별시 금천구 벚꽃로 298 1층 112호,37.4814401689835,126.883871693415
3,가산아스크타워점,서울특별시 금천구 가산동 60-9 145호,37.476704282032,126.889793917736
4,가산한화비즈2차점,"서울특별시 금천구 가산디지털2로 43-14, 가산한화비즈메트로2차 134호",37.4724347051782,126.881544629092
...,...,...,...,...
98,합정역점,서울특별시 마포구 양화로 72 서교 효성해링턴,37.5508138163956,126.916493990351
99,홈플러스강동점,서울특별시 강동구 양재대로1571 홈플러스3F,37.5457145865749,127.14225531407
100,홍대서교점,서울특별시 마포구 서교동 331-18,37.5561764842409,126.925834404005
101,홍대패션거리점,서울특별시 마포구 어울마당로 112-6,37.5546666402963,126.923369200074


In [6]:
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
csv_file = f"tencafe{timestamp}.csv"
df.to_csv(csv_file, index=True)
print("csv 변환완료")

hdfs = InsecureClient('http://namenode:9870',user='hadoop')
hdfs_path = f"/dataset/{csv_file}"
try:
    hdfs.upload(hdfs_path, csv_file)
    print("하둡에 업로드 완료!")
except:
    print("업로드 실패!")

csv 변환완료
하둡에 업로드 완료!
